# Phone Unlock State Machine Audit

This notebook inspects `data/android/unlock-events.jsonl` and checks whether the expected phone interval state machine is sane before importing it into Behaviour Hub.

The two candidate strategies are:

- `keyguard`: `KEYGUARD_HIDDEN` starts `Screen / Screen on`, `KEYGUARD_SHOWN` ends it.
- `screen`: `SCREEN_INTERACTIVE` starts `Screen / Screen on`, `SCREEN_NON_INTERACTIVE` ends it.

The importer currently repairs repeated starts by replacing the earlier open start with the later start.

In [ ]:
from __future__ import annotations

import json
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import polars as pl

from timeline.domain.enums import EventKind
from timeline.ingest.android import import_android_unlock_jsonl, parse_android_timestamp

In [ ]:
PATH = Path("../data/android/unlock-events.jsonl")

rows = []
with PATH.open(encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue
        row = json.loads(line)
        row["line_number"] = line_number
        row["parsed_timestamp"] = parse_android_timestamp(row["timestamp"])
        rows.append(row)

df = pl.DataFrame(rows)
df.select([
    pl.len().alias("rows"),
    pl.col("parsed_timestamp").min().alias("first_timestamp"),
    pl.col("parsed_timestamp").max().alias("last_timestamp"),
])

In [ ]:
(
    df
    .group_by(["event_type", "event"])
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
)

## State Machine Check

For each strategy, the desired interval stream is simply:

`START, END, START, END, ...`

Bad transitions are:

- `duplicate_start`: another start appears while an interval is already open.
- `orphan_end`: an end appears while no interval is open.
- `open_at_eof`: the file ends with an interval still open.

The importer repairs `duplicate_start` by replacing the earlier open start with the later one.

In [ ]:
STRATEGIES = {
    "keyguard": {
        "KEYGUARD_HIDDEN": EventKind.INTERVAL_START,
        "KEYGUARD_SHOWN": EventKind.INTERVAL_END,
    },
    "screen": {
        "SCREEN_INTERACTIVE": EventKind.INTERVAL_START,
        "SCREEN_NON_INTERACTIVE": EventKind.INTERVAL_END,
    },
}


@dataclass(frozen=True)
class StateMachineScan:
    strategy: str
    intervals: list[dict]
    anomalies: list[dict]
    ignored_events: int


def scan_state_machine(strategy: str, *, repair_duplicate_starts: bool) -> StateMachineScan:
    event_map = STRATEGIES[strategy]
    intervals = []
    anomalies = []
    ignored_events = 0
    open_start = None

    for row in rows:
        event_kind = event_map.get(row["event"])
        if event_kind is None:
            ignored_events += 1
            continue

        if event_kind == EventKind.INTERVAL_START:
            if open_start is not None:
                anomalies.append({
                    "kind": "duplicate_start",
                    "strategy": strategy,
                    "line_number": row["line_number"],
                    "timestamp": row["parsed_timestamp"],
                    "event": row["event"],
                    "open_line_number": open_start["line_number"],
                    "open_timestamp": open_start["parsed_timestamp"],
                    "open_event": open_start["event"],
                })
                if repair_duplicate_starts:
                    open_start = row
                continue

            open_start = row
            continue

        if open_start is None:
            anomalies.append({
                "kind": "orphan_end",
                "strategy": strategy,
                "line_number": row["line_number"],
                "timestamp": row["parsed_timestamp"],
                "event": row["event"],
            })
            continue

        intervals.append({
            "strategy": strategy,
            "start_line_number": open_start["line_number"],
            "end_line_number": row["line_number"],
            "start_timestamp": open_start["parsed_timestamp"],
            "end_timestamp": row["parsed_timestamp"],
            "duration_seconds": int((row["parsed_timestamp"] - open_start["parsed_timestamp"]).total_seconds()),
            "start_event": open_start["event"],
            "end_event": row["event"],
        })
        open_start = None

    if open_start is not None:
        anomalies.append({
            "kind": "open_at_eof",
            "strategy": strategy,
            "line_number": open_start["line_number"],
            "timestamp": open_start["parsed_timestamp"],
            "event": open_start["event"],
        })

    return StateMachineScan(
        strategy=strategy,
        intervals=intervals,
        anomalies=anomalies,
        ignored_events=ignored_events,
    )

In [ ]:
summary_rows = []
scans = {}

for strategy in STRATEGIES:
    raw_scan = scan_state_machine(strategy, repair_duplicate_starts=False)
    repaired_scan = scan_state_machine(strategy, repair_duplicate_starts=True)
    importer_result = import_android_unlock_jsonl(PATH, strategy=strategy)
    scans[(strategy, "raw")] = raw_scan
    scans[(strategy, "repaired")] = repaired_scan

    anomaly_counts = Counter(anomaly["kind"] for anomaly in raw_scan.anomalies)
    repaired_anomaly_counts = Counter(anomaly["kind"] for anomaly in repaired_scan.anomalies)
    summary_rows.append({
        "strategy": strategy,
        "raw_intervals": len(raw_scan.intervals),
        "repaired_intervals": len(repaired_scan.intervals),
        "ignored_events": raw_scan.ignored_events,
        "raw_duplicate_starts": anomaly_counts["duplicate_start"],
        "raw_orphan_ends": anomaly_counts["orphan_end"],
        "raw_open_at_eof": anomaly_counts["open_at_eof"],
        "repaired_duplicate_starts": repaired_anomaly_counts["duplicate_start"],
        "repaired_orphan_ends": repaired_anomaly_counts["orphan_end"],
        "repaired_open_at_eof": repaired_anomaly_counts["open_at_eof"],
        "importer_events": len(importer_result.events),
        "importer_dropped_orphan_starts": importer_result.dropped_orphan_starts,
    })

pl.DataFrame(summary_rows)

## Duration Audit

Large durations are usually where state-machine mistakes show up first.

In [ ]:
duration_rows = []

for strategy in STRATEGIES:
    for mode in ["raw", "repaired"]:
        intervals = scans[(strategy, mode)].intervals
        if not intervals:
            continue
        interval_df = pl.DataFrame(intervals)
        duration_rows.append({
            "strategy": strategy,
            "mode": mode,
            "intervals": interval_df.height,
            "total_hours": interval_df["duration_seconds"].sum() / 3600,
            "median_seconds": interval_df["duration_seconds"].median(),
            "p95_seconds": interval_df["duration_seconds"].quantile(0.95),
            "max_seconds": interval_df["duration_seconds"].max(),
        })

pl.DataFrame(duration_rows).sort(["strategy", "mode"])

In [ ]:
def longest_intervals(strategy: str, mode: str, n: int = 20) -> pl.DataFrame:
    intervals = scans[(strategy, mode)].intervals
    if not intervals:
        return pl.DataFrame()
    return (
        pl.DataFrame(intervals)
        .sort("duration_seconds", descending=True)
        .head(n)
    )

longest_intervals("keyguard", "raw", 20)

In [ ]:
def anomalies(strategy: str, mode: str = "raw") -> pl.DataFrame:
    items = scans[(strategy, mode)].anomalies
    return pl.DataFrame(items) if items else pl.DataFrame()

anomalies("keyguard")

## Inspect Rows Around Anomalies

Use this to inspect the raw event sequence around duplicate starts or orphan ends.

In [ ]:
def rows_around(line_number: int, radius: int = 6) -> pl.DataFrame:
    start = max(1, line_number - radius)
    end = line_number + radius
    return (
        df
        .filter(pl.col("line_number").is_between(start, end))
        .select([
            "line_number",
            "id",
            "parsed_timestamp",
            "event_type",
            "event",
            "package_name",
            "class_name",
        ])
    )

keyguard_anomalies = anomalies("keyguard")
if keyguard_anomalies.is_empty():
    print("No keyguard anomalies found.")
else:
    rows_around(keyguard_anomalies["line_number"][0])

## Importer Parity Check

These checks compare the notebook's repaired state-machine scan with the actual importer.

In [ ]:
for strategy in STRATEGIES:
    importer_result = import_android_unlock_jsonl(PATH, strategy=strategy)
    repaired_scan = scans[(strategy, "repaired")]
    repaired_event_count = len(repaired_scan.intervals) * 2
    if any(anomaly["kind"] == "open_at_eof" for anomaly in repaired_scan.anomalies):
        repaired_event_count += 1

    print(strategy)
    print("  importer events:", len(importer_result.events))
    print("  repaired scan events:", repaired_event_count)
    print("  importer dropped orphan starts:", importer_result.dropped_orphan_starts)
    print("  raw duplicate starts:", sum(1 for anomaly in scans[(strategy, "raw")].anomalies if anomaly["kind"] == "duplicate_start"))